<style>
/* RTL support for Colab */
div[dir="rtl"] {
  text-align: right;
  direction: rtl;
  font-family: Arial, "Segoe UI", Tahoma, sans-serif;
  line-height: 1.8;
}
div[dir="rtl"] h1, div[dir="rtl"] h2, div[dir="rtl"] h3 {
  text-align: right;
}
div[dir="rtl"] table {
  direction: rtl;
  margin-right: 0;
  margin-left: auto;
}
div[dir="rtl"] code {
  direction: ltr;
  display: inline-block;
}
div[dir="rtl"] pre {
  direction: ltr;
  text-align: left;
}
</style>
<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

#Smart Traffic System


### الفكرة العامة:
نظام ذكي يُثبَّت على كاميرا مراقبة فوق طريق مدفوع (Toll Road) ذي اتجاهين (ذهاب/إياب). يقوم النظام ب:
1. **كشف المركبات وتصنيفها** باستخدام YOLO11n
2. **عدّ المركبات** العابرة وتحديد اتجاهها (ذهاب/إياب)
3. **حساب رسم العبور**: سيارة $1، فان $1.5، باص $2، شاحنة $2.5
4. **منع الموتورات**: إرسال تنبيه فوري للشرطة عند مرور موتور مخالف
5. **مراقبة السرعة**: تمييز المركبة المخالفة بإطار أحمر
6. **التقرير النهائي**: العدد الكلي لكل نوع + الدخل المحصّل

### الشروط الأساسية:
- معالجة في الزمن الحقيقي
- دقة عالية حتى في حالات التداخل (Occlusion) بنسبة لا تقل عن 85%
- مرونة في ظروف الإضاءة والطقس
- استهلاك أمثل للموارد (No GPU - 8 CPU)
- استخدام YOLO11n تحديدًا (لا يُجوز إصدار آخر)

---

### الخريطة الزمنية للخطوات:

| الخطوة | الموضوع | التقنية |
|--------|---------|---------|
| 1 | تثبيت المكتبات | pip install |
| 2 | تنزيل الفيديو | yt-dlp + ffmpeg |
| 3 | المعالجة المسبقة | OpenCV (Gaussian Blur + CLAHE) |
| 4 | كشف المركبات | YOLO11n |
| 5 | خوارزمية التتبع | Centroid + IoU (Hungarian) |
| 6 | العد والاتجاه | Crossing Line + Cross Product |
| 7 | حساب السرعة | EMA من تاريخ المواضع |
| 8 | كشف المخالفات | موتور مخالف + سرعة |
| 9 | رسم التعليقات | Bounding Boxes + ألوان |
| 10 | التقرير النهائي | إحصائيات + دخل |


</div>

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 1: تثبيت المكتبات

نحتاج إلى:
- **ultralytics**: لتشغيل نموذج YOLO11n
- **opencv-python**: لمعالجة الفيديو وتطبيق الفلاتر والالتفاف (Convolution)
- **numpy**: للعمليات الحسابية ومصفوفات التكاليف
- **scipy**: للخوارزمية المجرية (Hungarian Algorithm) في التتبع
- **yt-dlp**: لتنزيل فيديو الاختبار من يوتيوب

</div>

In [ ]:
# تثبيت المكتبات المطلوبة
!pip install ultralytics opencv-python-headless scipy yt-dlp -q

print('تم تثبيت جميع المكتبات بنجاح')

In [ ]:
# استيراد المكتبات
import cv2
import numpy as np
from collections import OrderedDict, defaultdict
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
import time
import os

print('تم استيراد جميع المكتبات')
print(f'OpenCV version: {cv2.__version__}')
print(f'NumPy version: {np.__version__}')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 2: تنزيل فيديو الاختبار وقصّه
 الفيديو التجريبي على يوتيوب:
`https://youtu.be/wqctLW0Hb_0`

يُستخدم فقط المقطع من **الدقيقة 4 إلى الدقيقة 9** (5 دقائق).

سنستخدم `yt-dlp` للتنزيل، ثم `ffmpeg` لقص المقطع المطلوب.

</div>

In [ ]:
# تنزيل الفيديو من يوتيوب
# ملاحظة: إذا لم يعمل الرابط، يمكنك رفع الفيديو يدويًا إلى Colab

video_url = 'https://youtu.be/wqctLW0Hb_0'

# محاولة التنزيل
!yt-dlp -f 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best' \
    --merge-output-format mp4 \
    -o 'full_video.mp4' \
    '{video_url}' 2>/dev/null || echo 'تعذر التنزيل - يرجى رفع الفيديو يدويًا'

# التحقق من وجود الفيديو
if os.path.exists('full_video.mp4'):
    print('تم تنزيل الفيديو الكامل')
else:
    print('لم يتم تنزيل الفيديو. يمكنك رفعه يدويًا وتسميته full_video.mp4')

In [ ]:
# قص المقطع من الدقيقة 4 إلى الدقيقة 9 باستخدام ffmpeg
# الدقيقة 4 = 00:04:00 ، الدقيقة 9 = 00:09:00 (المدة 5 دقائق)

if os.path.exists('full_video.mp4'):
    !ffmpeg -y -ss 00:04:00 -to 00:09:00 -i full_video.mp4 \
        -c:v libx264 -c:a aac \
        test_video.mp4 2>/dev/null

    if os.path.exists('test_video.mp4'):
        cap = cv2.VideoCapture('test_video.mp4')
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        print(f'تم قص الفيديو: {width}x{height} | {fps:.1f} FPS | {total} إطار')
    else:
        print('تعذر قص الفيديو')
else:
    print('الفيديو الكامل غير موجود')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 3: إعداد ملف التكوين (Config)

هنا نُعرّف كل إعدادات المشروع:
- أسعار رسم العبور لكل نوع مركبة
- ألوان المربعات (BGR) لكل فئة (الأحمر محجوز للمخالفات)
- خطوط العبور (Crossing Lines) لكل اتجاه
- الحد الأقصى للسرعة
- إعدادات فلاتر المعالجة المسبقة
- تعيين فئات COCO إلى فئات المشروع


</div>

In [ ]:
# ============================================================
# ملف الإعدادات الرئيسي
# ============================================================

# --- إعدادات الفيديو ---
VIDEO_PATH = 'test_video.mp4'
OUTPUT_PATH = 'output_video.mp4'

# --- إعدادات النموذج ---
# yolo11n.pt للنموذج الجاهز (MVP)
# best.pt بعد Fine-tuning على الفئات الست
MODEL_WEIGHTS = 'yolo11n.pt'
CONF_THRESHOLD = 0.4      # عتبة الثقة
IOU_THRESHOLD = 0.5       # عتبة IoU لـ NMS

# --- تعيين فئات COCO إلى فئات المشروع ---
# النموذج الجاهز يدعم: car=2, motorcycle=3, bus=5, truck=7
# الفان وموتور الشرطة تحتاج Fine-tuning
COCO_TO_PROJECT = {
    2: 'car',        # سيارة
    3: 'motorcycle', # موتور
    5: 'bus',        # باص
    7: 'truck',      # شاحنة
}

# فئات المشروع الست (بعد Fine-tuning)
PROJECT_CLASSES = [
    'car',                    # سيارة
    'van',                    # فان
    'bus',                    # باص
    'truck',                  # شاحنة
    'motorcycle',             # موتور
    'traffic_police_motorcycle' # موتور شرطة
]

# --- أسعار رسم العبور (دولار) ---
TOLL_PRICES = {
    'car':                       1.0,
    'van':                       1.5,
    'bus':                       2.0,
    'truck':                     2.5,
    'motorcycle':                0.0,  # ممنوع
    'traffic_police_motorcycle':  0.0,  # معفى
}

# --- ألوان المربعات (BGR) ---
# الأحمر (0,0,255) محجوز للمخالفات فقط
CLASS_COLORS = {
    'car':                       (255, 0, 0),     # أزرق
    'van':                       (0, 200, 255),   # برتقالي
    'bus':                       (0, 255, 255),   # أصفر
    'truck':                     (0, 255, 0),     # أخضر
    'motorcycle':                (255, 0, 255),   # أرجواني
    'traffic_police_motorcycle':  (255, 255, 0),  # سماوي
}
VIOLATION_COLOR = (0, 0, 255)  # أحمر للمخالفات

# --- خطوط العبور (Crossing Lines) ---
# كل خط خاص باتجاه: going (ذهاب) / returning (إياب)
# عدّل هذه الإحداثيات حسب موضع الكاميرا في الفيديو
CROSSING_LINE = {
    'going': {       # خط الذهاب
        'p1': (0, 350),
        'p2': (640, 350),
    },
    'returning': {   # خط الإياب
        'p1': (0, 450),
        'p2': (640, 450),
    },
}

# منطقة الهستيريسيس: تجاهل النقاط قريبة جدًا من الخط
HYSTERESIS_PIXELS = 10

# --- إعدادات السرعة ---
MAX_SPEED_KMH = 80                # الحد الأقصى (كم/ساعة)
METERS_PER_PIXEL = 0.05           # معامل تحويل (تقديري - يحتاج معايرة)
SPEED_SMOOTH_FRAMES = 5            # عدد الإطارات لحساب متوسط السرعة (EMA)

# --- إعدادات التتبع ---
MAX_DISAPPEARED = 30              # إطارات قبل حذف المسار
MAX_DISTANCE = 80                 # أقصى مسافة لربط الكشف بالمسار

# --- إعدادات المعالجة المسبقة ---
USE_GAUSSIAN_BLUR = True
GAUSSIAN_KERNEL = (5, 5)
USE_CLAHE = True
CLAHE_CLIP_LIMIT = 2.0
CLAHE_GRID_SIZE = (8, 8)

# --- إعدادات التنبيهات ---
ALERTS_FILE = 'alerts.txt'
DIRECTION_TO_SIDE = {
    'going':     'يمين',
    'returning': 'يسار',
}

print('تم تحميل جميع الإعدادات')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 4: المعالجة المسبقة (Preprocessing) باستخدام OpenCV

 استخدام Python & OpenCV لتطبيق الفلاتر (Filters) والالتفاف (Convolution) في تحسين جودة الفيديو وإزالة الضوضاء (Noise Reduction).

### الفلاتر المستخدمة:

#### 1. Gaussian Blur (مرشّح غاوسي)
مرشّح التفافي (Convolution Filter) يطبّق نواة غاوسية على كل بكسل.
- يزيل الضوضاء عالية التردد مع الحفاظ على الحواف العامة
- النواة: مصفوفة 5x5 تمرّر على الصورة
- المعادلة: G(x,y) = (1/2*pi*sigma^2) * exp(-(x^2+y^2)/(2*sigma^2))

#### 2. CLAHE (Contrast Limited Adaptive Histogram Equalization)
يعزز التباين محليًا في مناطق صغيرة من الصورة.
- مفيد جدًا لتغيرات الإضاءة (نهار/ليل)
- يقسّم الصورة إلى شبكة (8x8)
- يطبّق معادلة المدرج التكراري (Histogram Equalization) على كل خلية
- يحدّ من التباين (Clip Limit) لمنع فرط التحسين

> **لماذا نطبّق المعالجة المسبقة قبل YOLO؟**
> تحسين جودة الإطار يزيد من دقة الكشف، خاصة في ظروف الإضاءة الضعيفة أو وجود ضباب/غبار.

</div>

In [ ]:
def preprocess_frame(frame):
    """
    تطبيق فلاتر OpenCV لتحسين جودة الإطار قبل الكشف.

    الخطوات:
    1. Gaussian Blur: إزالة الضوضاء (Noise Reduction) عبر الالتفاف (Convolution)
    2. CLAHE: تحسين التباين التكيفي (مفيد لتغيرات الإضاءة)
    """
    # 1. Gaussian Blur - إزالة الضوضاء
    # نواة 5x5: كل بكسل يُستبدل بالمتوسط المرجّح لجيرانه
    if USE_GAUSSIAN_BLUR:
        frame = cv2.GaussianBlur(frame, GAUSSIAN_KERNEL, 0)

    # 2. CLAHE - تحسين التباين التكيفي
    # نحوّل إلى LAB color space لتحسين قناة الإضاءة (L) فقط
    if USE_CLAHE:
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=CLAHE_CLIP_LIMIT,
            tileGridSize=CLAHE_GRID_SIZE,
        )
        l_channel = clahe.apply(l_channel)
        lab = cv2.merge((l_channel, a_channel, b_channel))
        frame = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    return frame

# اختبار المعالجة المسبقة على إطار واحد
if os.path.exists(VIDEO_PATH):
    cap_test = cv2.VideoCapture(VIDEO_PATH)
    ret, test_frame = cap_test.read()
    if ret:
        processed = preprocess_frame(test_frame.copy())
        cv2.imwrite('original_frame.jpg', test_frame)
        cv2.imwrite('processed_frame.jpg', processed)
        print('تم تطبيق المعالجة المسبقة على إطار تجريبي')
    cap_test.release()
else:
    print('الفيديو غير متوفر للتطبيق التجريبي')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 5: تحميل نموذج YOLO11n

- **YOLO11n** هو أصغر وأسرع نموذج في عائلة YOLO11
- مناسب للأجهزة متوسطة القدرة (No GPU - 8 CPU)
- سرعة عالية في الكشف (Real-time)

### الفئات المدعومة في النموذج الجاهز (pretrained):
النموذج مدرّب على مجموعة COCO التي تحتوي على 80 فئة، منها:

| رقم COCO | الاسم | فئة المشروع |
|----------|------|-------------|
| 2 | car | سيارة |
| 3 | motorcycle | موتور |
| 5 | bus | باص |
| 7 | truck | شاحنة |

</div>

In [ ]:
# تحميل نموذج YOLO11n الجاهز (pretrained)
# سيتم تنزيله تلقائيًا عند أول تشغيل
print('تحميل نموذج YOLO11n...')
model = YOLO(MODEL_WEIGHTS)
print('تم تحميل النموذج بنجاح')

# عرض الفئات المدعومة
print('\nفئات النموذج (COCO) المرتبطة بالمشروع:')
for cls_id, name in model.names.items():
    if cls_id in COCO_TO_PROJECT:
        print(f'  {cls_id}: {name} -> {COCO_TO_PROJECT[cls_id]} (فئة المشروع)')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 6: دالة الكشف عن المركبات (Detection)

هذه الدالة تأخذ إطارًا معالجًا وتمرّره إلى YOLO11n، ثم تستخرج المربعات (Bounding Boxes) وأسماء الفئات.

### معالجة خاصة:
- إذا كان النموذج الجاهز (yolo11n.pt): نطبّق تعيين COCO إلى فئات المشروع
- إذا كان النموذج المدرّب (best.pt): نقرأ أسماء الفئات مباشرة من النموذج
- نتجاهل أي كشف لا ينتمي لفئات المركبات الست

### عتبة الثقة (Confidence Threshold):
نقبل فقط الكشوفات بثقة >= 0.4 (40%). هذا يقلل الإيجابيات الكاذبة (False Positives).

</div>

In [ ]:
def detect_vehicles(frame, model):
    """
    كشف المركبات باستخدام YOLO11n.
    يدعم النموذج الجاهز (yolo11n.pt) والنموذج المدرّب (best.pt).
    """
    results = model(
        frame,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False,
    )

    model_names = model.names if hasattr(model, 'names') else {}

    detections = []
    for result in results:
        boxes = result.boxes
        for box in boxes:
            cls_id = int(box.cls[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # 1. قراءة اسم الفئة من النموذج (يدعم best.pt)
            class_name = None
            if model_names and cls_id in model_names:
                raw_name = str(model_names[cls_id]).strip().lower()
                normalized = raw_name.replace(' ', '_').replace('-', '_')
                if normalized in PROJECT_CLASSES:
                    class_name = normalized
                elif normalized in COCO_TO_PROJECT.values():
                    class_name = normalized

            # 2. Fallback: تعيين من COCO IDs
            if class_name is None and cls_id in COCO_TO_PROJECT:
                class_name = COCO_TO_PROJECT[cls_id]

            # 3. تجاهل الفئات غير المتعلقة بالمركبات
            if class_name is None:
                continue

            detections.append(([x1, y1, x2, y2], class_name))

    return detections

# اختبار الكشف
if os.path.exists('processed_frame.jpg'):
    test_img = cv2.imread('processed_frame.jpg')
    dets = detect_vehicles(test_img, model)
    print(f'تم الكشف عن {len(dets)} مركبة في الإطار التجريبي')
    for bbox, cls in dets:
        print(f'   - {cls}: bbox={bbox}')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 7: خوارزمية التتبع (Tracking)

### لماذا نحتاج التتبع؟
YOLO يكشف المركبات في كل إطار بشكل مستقل، لكنه لا يتتبّع هوية المركبة عبر الإطارات. بدون تتبع:
- قد نعدّ نفس المركبة عدة مرات
- لا نستطيع حساب السرعة (تحتاج تاريخ المواضع)
- فقدان الهوية أثناء التداخل (Occlusion)

### خوارزمية Centroid Tracker مع IoU:

1. **لكل مركبة**: نحسب النقطة المركزية (Centroid) = مركز المربع
2. **المطابقة**: نربط كل كشف جديد بأقرب مسار حالي باستخدام:
   - المسافة الأوروبية بين المراكز
   - IoU (Intersection over Union) بين المربعات
   - الخوارزمية المجرية (Hungarian Algorithm) للمطابقة المثلى
3. **التسجيل**: الكشوفات غير المطابقة -> مسارات جديدة
4. **الإلغاء**: المسارات غير المرئية لـ 30 إطارًا -> تُحذف

### الصيغة الرياضية لـ IoU:
IoU = |A ∩ B| / |A ∪ B|

حيث A و B هما المربعان. IoU = 1 يعني تطابق تام، IoU = 0 يعني لا تداخل.

### لماذا نحتفظ بالمسارات المفقودة (max_disappeared=30)؟
عندما تتداخل مركبتان (Occlusion)، قد يفقد YOLO إحداهما لعدة إطارات.
بدلاً من حذفها فورًا، نحتفظ بها لـ 30 إطارًا. إذا عادت للظهور، نربطها بنفس الهوية.
هذا يساعد في تحقيق شرط **85% من حالات التداخل**.

</div>

In [ ]:
class CentroidTracker:
    """
    متتبّع يعتمد على النقاط المركزية (Centroids) مع مطابقة IoU.
    يستخدم الخوارزمية المجرية (Hungarian Algorithm) للمطابقة المثلى.
    """

    def __init__(self, max_disappeared=30, max_distance=80):
        self.next_object_id = 0
        self.objects = OrderedDict()
        self.disappeared = OrderedDict()
        self.max_disappeared = max_disappeared
        self.max_distance = max_distance
        self.position_history = OrderedDict()  # للسرعة

    def register(self, centroid, bbox, class_name):
        """تسجيل مركبة جديدة"""
        self.objects[self.next_object_id] = {
            'centroid': centroid,
            'bbox': bbox,
            'class': class_name,
            'speed': 0.0,
        }
        self.disappeared[self.next_object_id] = 0
        self.position_history[self.next_object_id] = [centroid]
        self.next_object_id += 1

    def deregister(self, object_id):
        """حذف مركبة"""
        del self.objects[object_id]
        del self.disappeared[object_id]
        if object_id in self.position_history:
            del self.position_history[object_id]

    def _compute_iou(self, boxA, boxB):
        """حساب IoU بين مربعين"""
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])
        interW = max(0, xB - xA)
        interH = max(0, yB - yA)
        interArea = interW * interH
        areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
        areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
        union = areaA + areaB - interArea
        return interArea / union if union > 0 else 0

    def update(self, detections):
        """تحديث المسارات بناءً على الكشوفات الجديدة."""
        if len(detections) == 0:
            for obj_id in list(self.disappeared.keys()):
                self.disappeared[obj_id] += 1
                if self.disappeared[obj_id] > self.max_disappeared:
                    self.deregister(obj_id)
            return self.objects

        new_centroids = []
        new_bboxes = []
        new_classes = []
        for (bbox, class_name) in detections:
            cx = int((bbox[0] + bbox[2]) / 2.0)
            cy = int((bbox[1] + bbox[3]) / 2.0)
            new_centroids.append((cx, cy))
            new_bboxes.append(bbox)
            new_classes.append(class_name)

        if len(self.objects) == 0:
            for i in range(len(new_centroids)):
                self.register(new_centroids[i], new_bboxes[i], new_classes[i])
            return self.objects

        object_ids = list(self.objects.keys())
        object_centroids = [self.objects[obj_id]['centroid'] for obj_id in object_ids]
        object_bboxes = [self.objects[obj_id]['bbox'] for obj_id in object_ids]

        n_objects = len(object_ids)
        n_detections = len(new_centroids)
        cost_matrix = np.zeros((n_objects, n_detections))

        for i in range(n_objects):
            for j in range(n_detections):
                dist = np.sqrt(
                    (object_centroids[i][0] - new_centroids[j][0]) ** 2 +
                    (object_centroids[i][1] - new_centroids[j][1]) ** 2
                )
                iou = self._compute_iou(object_bboxes[i], new_bboxes[j])
                cost_matrix[i, j] = dist - (iou * 50)

        row_indices, col_indices = linear_sum_assignment(cost_matrix)

        used_rows = set()
        used_cols = set()

        for (row, col) in zip(row_indices, col_indices):
            if cost_matrix[row, col] <= self.max_distance:
                obj_id = object_ids[row]
                self.objects[obj_id]['centroid'] = new_centroids[col]
                self.objects[obj_id]['bbox'] = new_bboxes[col]
                self.objects[obj_id]['class'] = new_classes[col]
                self.disappeared[obj_id] = 0
                self.position_history[obj_id].append(new_centroids[col])
                if len(self.position_history[obj_id]) > 30:
                    self.position_history[obj_id].pop(0)
                used_rows.add(row)
                used_cols.add(col)

        for row in range(n_objects):
            if row not in used_rows:
                obj_id = object_ids[row]
                self.disappeared[obj_id] += 1
                if self.disappeared[obj_id] > self.max_disappeared:
                    self.deregister(obj_id)

        for col in range(n_detections):
            if col not in used_cols:
                self.register(new_centroids[col], new_bboxes[col], new_classes[col])

        return self.objects

    def get_speed(self, obj_id, fps=30, meters_per_pixel=0.05, smooth_frames=5):
        """تقدير السرعة من متوسط حركة آخر N إطارات (EMA)."""
        if obj_id not in self.position_history:
            return 0.0
        history = self.position_history[obj_id]
        if len(history) < 2:
            return 0.0
        recent = history[-(smooth_frames + 1):] if len(history) > smooth_frames else history
        if len(recent) < 2:
            return 0.0
        total_dist = 0.0
        for i in range(1, len(recent)):
            dx = recent[i][0] - recent[i - 1][0]
            dy = recent[i][1] - recent[i - 1][1]
            total_dist += np.sqrt(dx ** 2 + dy ** 2)
        avg_pixel_per_frame = total_dist / (len(recent) - 1)
        speed_ms = avg_pixel_per_frame * meters_per_pixel * fps
        speed_kmh = speed_ms * 3.6
        return speed_kmh

print('تم تعريف خوارزمية التتبع (CentroidTracker)')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 8: منطق العد والاتجاه (Counting & Direction)

### الفكرة:
نحدد **خط عبور** (Crossing Line) لكل اتجاه. عندما تعبر مركبة الخط، نعرف:
1. **أنها عبرت** -> نعدها
2. **أي خط عبرت** -> نحدد الاتجاه (ذهاب/إياب)

### كيف نكتشف العبور؟
نستخدم **الضرب المتجهي (Cross Product)** لتحديد جهة المركبة بالنسبة للخط:

cross = (x2-x1)(yp-y1) - (y2-y1)(xp-x1)

- إذا cross > 0 -> المركبة في جهة 'above'
- إذا cross < 0 -> المركبة في جهة 'below'

عندما تتغير الجهة من above إلى below (أو العكس)، نعرف أن المركبة عبرت الخط.

### منع الازدواج في العد:
- نحتفظ بمجموعة `counted_ids` لكل المركبات التي تم عدّها
- كل مركبة تُعدّ **مرة واحدة فقط** في حياتها

### الهستيريسيس (Hysteresis):
عندما تكون المركبة قريبة جدًا من الخط (داخل هامش 10 بيكسل)، لا نحدّث جهتها.
هذا يمنع **اهتزاز العد** عندما تتذبذب المركبة حول الخط.

</div>

In [ ]:
def point_side(point, line_p1, line_p2):
    """تحديد جهة النقطة بالنسبة للخط (Cross Product)."""
    cross = (line_p2[0] - line_p1[0]) * (point[1] - line_p1[1]) - \
            (line_p2[1] - line_p1[1]) * (point[0] - line_p1[0])
    return 'above' if cross > 0 else 'below'

def signed_distance(point, line_p1, line_p2):
    """المسافة الموقّعة من النقطة إلى الخط (للهستيريسيس)."""
    dx = line_p2[0] - line_p1[0]
    dy = line_p2[1] - line_p1[1]
    length = np.sqrt(dx ** 2 + dy ** 2)
    if length == 0:
        return 0.0
    cross = dx * (point[1] - line_p1[1]) - dy * (point[0] - line_p1[0])
    return cross / length

class CountingState:
    """حالة العد والتتبع عبر الإطارات"""
    def __init__(self):
        self.counts = defaultdict(lambda: defaultdict(int))
        self.counted_ids = set()
        self.counted_ids_dir = {}
        self.previous_side = {}
        self.alerts = []
        self.motorcycle_alerted = set()
        self.speed_violation_ids = set()

    def check_crossing(self, obj_id, centroid, class_name):
        """فحص عبور المركبة لخط العبور وعدّها مرة واحدة."""
        direction = None
        for line_key, line in CROSSING_LINE.items():
            p1 = line['p1']
            p2 = line['p2']
            current_side = point_side(centroid, p1, p2)
            signed_dist = signed_distance(centroid, p1, p2)

            prev_key = (obj_id, line_key)
            if prev_key in self.previous_side:
                prev = self.previous_side[prev_key]
                # الهستيريسيس: تجاهل النقاط قريبة جدًا من الخط
                if abs(signed_dist) < HYSTERESIS_PIXELS:
                    continue
                if prev != current_side:
                    if obj_id not in self.counted_ids:
                        direction = line_key
                        self.counted_ids.add(obj_id)
                        self.counted_ids_dir[obj_id] = direction
                        self.counts[direction][class_name] += 1
                        print(f'[COUNT] {class_name} عبرت خط {direction} '
                              f'(ID: {obj_id}) - العدد: {self.counts[direction][class_name]}')

            if abs(signed_dist) >= HYSTERESIS_PIXELS:
                self.previous_side[prev_key] = current_side

        return direction

print('تم تعريف منطق العد والاتجاه')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 9: كشف المخالفات (Violations)

### نوعان من المخالفات:

### 1. مخالفة مرور موتور (Motorcycle Violation)
الملف يصرّح: "يُمنع مرور الدراجات النارية (الموتورات) على هذا الطريق باستثناء موتورات شرطة المرور"

عند رصد موتور غير تابع للشرطة، يجب إرسال إشعار فوري:
```
!انتبه مرور مخالف قادم إليك - يمين/شمال
```
جهة القدوم (يمين/يسار) تُحدد من اتجاه عبور خط العبور.

### 2. مخالفة سرعة (Speed Violation)
إذا تجاوزت سرعة أي مركبة الحد الأقصى المحدد، يُميّز إطارها باللون الأحمر.

</div>

In [ ]:
def check_motorcycle_violation(state, obj_id, class_name, direction, alerts_file):
    """فحص مرور موتور غير تابع للشرطة وإرسال تنبيه فوري."""
    if class_name == 'motorcycle':
        if obj_id not in state.motorcycle_alerted:
            state.motorcycle_alerted.add(obj_id)
            side = DIRECTION_TO_SIDE.get(direction, 'يمين')
            alert_msg = f'!انتبه مرور مخالف قادم إليك - {side}'
            timestamp = time.strftime('%H:%M:%S')
            full_alert = f'[{timestamp}] {alert_msg} (ID: {obj_id}, الاتجاه: {direction})'
            state.alerts.append(full_alert)
            alerts_file.write(full_alert + '\n')
            alerts_file.flush()
            print(f'[ALERT] {full_alert}')
            return alert_msg
    return None

print('تم تعريف كشف المخالفات')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 10: رسم التعليقات التوضيحية (Annotations)

### قواعد الألوان :
- **كل نوع مركبة له لون مختلف** يميّزه في الفيديو
- **اللون الأحمر محجوز حصرًا للمركبات المخالفة**

| الفئة | اللون | كود BGR |
|-------|-------|---------|
| سيارة (car) | أزرق | (255, 0, 0) |
| فان (van) | برتقالي | (0, 200, 255) |
| باص (bus) | أصفر | (0, 255, 255) |
| شاحنة (truck) | أخضر | (0, 255, 0) |
| موتور (motorcycle) | أرجواني | (255, 0, 255) |
| موتور شرطة | سماوي | (255, 255, 0) |
| **مخالفة** | **أحمر** | **(0, 0, 255)** |

### ما يُعرض على كل مربع:
- اسم الفئة + رقم التعريف (ID) + السرعة المقدّرة
- علامة [SPEED!] عند مخالفة السرعة
- علامة [VIOLATION!] عند موتور مخالف

</div>

In [ ]:
def draw_annotations(frame, objects, tracker, state, real_fps=0.0):
    """رسم المربعات والأسماء والسرعة. الأحمر للمخالفات فقط."""
    for obj_id, obj in objects.items():
        bbox = obj['bbox']
        class_name = obj['class']
        cx, cy = obj['centroid']

        x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])

        speed = tracker.get_speed(
            obj_id, fps=real_fps if real_fps > 0 else 30,
            meters_per_pixel=METERS_PER_PIXEL,
            smooth_frames=SPEED_SMOOTH_FRAMES,
        )

        is_speed_violation = speed > MAX_SPEED_KMH
        is_motorcycle_violation = class_name == 'motorcycle'

        if is_speed_violation or is_motorcycle_violation:
            color = VIOLATION_COLOR
        else:
            color = CLASS_COLORS.get(class_name, (255, 255, 255))

        thickness = 3 if (is_speed_violation or is_motorcycle_violation) else 2
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

        label = f'{class_name} ID:{obj_id} {speed:.0f}km/h'
        if is_speed_violation:
            label += ' [SPEED!]'
        if is_motorcycle_violation:
            label += ' [VIOLATION!]'

        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw, y1), color, -1)
        cv2.putText(frame, label, (x1, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.circle(frame, (cx, cy), 4, color, -1)

    for line_key, line in CROSSING_LINE.items():
        p1, p2 = line['p1'], line['p2']
        color = (0, 255, 255) if line_key == 'going' else (255, 200, 0)
        cv2.line(frame, p1, p2, color, 2)
        cv2.putText(frame, line_key, (p1[0] + 10, p1[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    if real_fps > 0:
        cv2.putText(frame, f'FPS: {real_fps:.1f}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    y_offset = 60
    for direction in ['going', 'returning']:
        for cls in PROJECT_CLASSES:
            cnt = state.counts[direction][cls]
            if cnt > 0:
                text = f'{direction} - {cls}: {cnt}'
                cv2.putText(frame, text, (10, y_offset),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                y_offset += 20

    return frame

print('تم تعريف رسم التعليقات التوضيحية')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 11: الحلقة الرئيسية (Main Loop)

### تسلسل المعالجة لكل إطار:
1. **قراءة الإطار** من الفيديو
2. **المعالجة المسبقة**: Gaussian Blur + CLAHE
3. **الكشف**: YOLO11n يكتشف المركبات
4. **التتبع**: Centroid Tracker يربط الكشوفات بالمسارات
5. **العد والاتجاه**: فحص عبور خط العبور
6. **كشف المخالفات**: موتور مخالف + سرعة
7. **الرسم**: المربعات والألوان
8. **الكتابة**: حفظ الإطار في الفيديو الناتج

</div>

In [ ]:
def run_traffic_system(video_path, model):
    """تشغيل نظام المرور الذكي على الفيديو."""
    print(f'[INFO] فتح الفيديو: {video_path}')

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'[ERROR] تعذر فتح الفيديو')
        return None, 0

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f'[INFO] FPS: {fps:.1f} | الدقة: {frame_width}x{frame_height} | إجمالي: {total_frames}')

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (frame_width, frame_height))

    tracker = CentroidTracker(MAX_DISAPPEARED, MAX_DISTANCE)
    state = CountingState()
    alerts_file = open(ALERTS_FILE, 'w', encoding='utf-8')
    alerts_file.write('=== سجل تنبيهات نظام المرور الذكي ===\n\n')

    start_time = time.time()
    frame_count = 0
    prev_time = time.time()
    fps_counter = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            print('[INFO] انتهى الفيديو.')
            break

        frame_count += 1

        # 1. المعالجة المسبقة
        processed = preprocess_frame(frame)

        # 2. الكشف
        detections = detect_vehicles(processed, model)

        # 3. التتبع
        objects = tracker.update(detections)

        # 4. العد والاتجاه + المخالفات
        for obj_id, obj in objects.items():
            direction = state.check_crossing(
                obj_id, obj['centroid'], obj['class']
            )
            if direction:
                check_motorcycle_violation(
                    state, obj_id, obj['class'], direction, alerts_file
                )
            speed = tracker.get_speed(
                obj_id, fps=fps,
                meters_per_pixel=METERS_PER_PIXEL,
                smooth_frames=SPEED_SMOOTH_FRAMES,
            )
            if speed > MAX_SPEED_KMH:
                state.speed_violation_ids.add(obj_id)

        # 5. الرسم
        frame = draw_annotations(frame, objects, tracker, state, fps)

        # FPS
        fps_counter += 1
        now = time.time()
        if now - prev_time >= 1.0:
            fps_display = fps_counter / (now - prev_time)
            fps_counter = 0
            prev_time = now

        out.write(frame)

        if frame_count % 100 == 0:
            print(f'  [إطار {frame_count}/{total_frames}] | مركبات نشطة: {len(objects)} | معدودة: {len(state.counted_ids)}')

    cap.release()
    out.release()
    elapsed = time.time() - start_time
    avg_fps = frame_count / elapsed if elapsed > 0 else 0

    print(f'\n[INFO] المعالجة اكتملت في {elapsed:.1f} ثانية')
    print(f'[INFO] متوسط FPS: {avg_fps:.1f}')

    alerts_file.write(f'\nعدد مخالفات الموتور: {len(state.alerts)}\n')
    alerts_file.write(f'عدد مخالفات السرعة: {len(state.speed_violation_ids)}\n')
    alerts_file.close()

    return state, avg_fps

print('تم تعريف الحلقة الرئيسية')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 12: تشغيل النظام

الآن نشغّل النظام على الفيديو.

> **ملاحظة**: في Colab، لا يمكن عرض نافذة الفيديو مباشرة. لذلك نحفظ الفيديو الناتج ونعرض لقطات منه.

</div>

In [ ]:
# تشغيل النظام
if os.path.exists(VIDEO_PATH):
    state, avg_fps = run_traffic_system(VIDEO_PATH, model)
else:
    print('الفيديو غير موجود. تأكد من تنزيله في الخطوة 2.')
    state = None

In [ ]:
# عرض لقطة من الفيديو الناتج
from google.colab.patches import cv2_imshow

if os.path.exists(OUTPUT_PATH):
    cap = cv2.VideoCapture(OUTPUT_PATH)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, snapshot = cap.read()
    if ret:
        cv2_imshow(snapshot)
        print('لقطة من الفيديو الناتج')
    cap.release()
else:
    print('الفيديو الناتج غير موجود')

<div dir="rtl" style="font-family: Arial, sans-serif; text-align: right;">

---
## الخطوة 13: التقرير النهائي

</div>

In [ ]:
def generate_final_report(state, avg_fps=0.0):
    """إصدار التقرير النهائي."""
    if state is None:
        print('لا توجد بيانات للتقرير')
        return

    total_revenue = 0.0
    total_vehicles = 0
    report_lines = []

    print('\n' + '=' * 60)
    print('           التقرير النهائي - نظام المرور الذكي')
    print('=' * 60)

    for direction in ['going', 'returning']:
        dir_name = 'ذهاب' if direction == 'going' else 'إياب'
        print(f'\n--- اتجاه {dir_name} ---')

        dir_total = 0
        for class_name in PROJECT_CLASSES:
            count = state.counts[direction][class_name]
            if count > 0:
                price = TOLL_PRICES.get(class_name, 0)
                revenue = count * price
                dir_total += revenue
                total_vehicles += count
                print(f'  {class_name:30s}: {count:4d} مركبة  |  رسم: ${revenue:.2f}')

        print(f'  {"المجموع الفرعي":30s}: ${dir_total:.2f}')
        total_revenue += dir_total

    print(f'\n{"=" * 60}')
    print(f'  إجمالي المركبات المارة: {total_vehicles}')
    print(f'  إجمالي الدخل المحصّل: ${total_revenue:.2f}')
    print(f'  عدد تنبيهات الموتور المخالف: {len(state.alerts)}')
    print(f'  عدد مخالفات السرعة: {len(state.speed_violation_ids)}')
    if avg_fps > 0:
        print(f'  متوسط سرعة المعالجة: {avg_fps:.1f} FPS')
    print(f'{"=" * 60}\n')

# إصدار التقرير
if state is not None:
    generate_final_report(state, avg_fps if 'avg_fps' in dir() else 0.0)

In [ ]:
# عرض محتوى ملف التنبيهات
if os.path.exists(ALERTS_FILE):
    print('=== محتوى ملف التنبيهات ===\n')
    with open(ALERTS_FILE, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print('ملف التنبيهات غير موجود')

In [ ]:
# تحميل الفيديو الناتج من Colab
from google.colab import files

if os.path.exists(OUTPUT_PATH):
    print('تحضير الفيديو الناتج للتحميل...')
    files.download(OUTPUT_PATH)
else:
    print('الفيديو الناتج غير موجود')